# Monitoring variant: r2lab/r2lab-2du/test.ipynb

Based on the upstream example. The original remains under `examples/r2lab/test.ipynb`.
Core: **open5gs**; RAN: **oai**. Stock Open5GS images are used; legacy custom core KPIs are disabled. The OAI monolithic-gNB KPI calculator is disabled for this RAN layout.

# **5G Deployment Example With R2lab:**

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UDM, UDR, UPF
- OAI split RAN: CU-CP, CU-UP and 2 DUs: `du01` with AW2S Jaguar in R2Lab and `du02` with AW2S Panther in R2Lab
- OAI FlexRIC
- COTS UEs: `qhat01`, `qhat02` and `qhat03` in R2Lab
- CSI: both DUs log SRS channel estimates per RB, shown live on `sopnode-f3` and collected per traffic section

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Install the Post-5G orchestrator:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
# CSI support is in this repository (branch csi): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

The Post-5G orchestrator keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  jaguar:
    type: real
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

  panther:
    type: real
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

ran:
  components:
    cucp:
      type: cu_cp
      config:
        ric: flexric

    cuup:
      type: cu_up
      config:
        cu_cp: cucp
        ric: flexric
  
    du01:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 50
          pointAFrequencyArfcn: 626748
          ssbArfcn: 627360
          ssbPbchBlockPowerDbm: -10
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 2
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        gnbDuId: 1
        nrCellId: 1
        physicalCellId: 1
        ru: jaguar
        cu_cp: cucp
        cu_up: cuup
        ric: flexric

    du02:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 50
          pointAFrequencyArfcn: 626748
          ssbArfcn: 627360
          ssbPbchBlockPowerDbm: -10
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 2
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        gnbDuId: 2
        nrCellId: 2
        physicalCellId: 2
        ru: panther
        cu_cp: cucp
        cu_up: cuup
        ric: flexric

ric:
  components:
    flexric:
      type: ric

ues:
  qhat01:
    type: cots
    config:
      supi: "001010000000006"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01

  qhat02:
    type: cots
    config:
      supi: "001010000000007"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01

  qhat03:
    type: cots
    config:
      supi: "001010000000008"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01
"""
)

To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

In this case, we are going to use `sopnode-f1` and `sopnode-f2` that are still currently on the legacy POS, and `sopnode-f3` on the new POS.

For the radio hardware, we are going to use `jaguar` and `panther` as RUs in R2Lab, as well as `qhat01`, `qhat02` and `qhat03` as UEs in R2Lab.

For that, it is necessary that you change the values in between `<...>` to contain your actual [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    extra_path: "" # Use SLICES CLI from PATH

provisioners:
  pos:
    type: pos
    params:
      attempt_reservation: true

  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>

  r2lab:
    type: r2lab
    params:
      r2lab_slice: <YOUR_R2LAB_SLICE>
      reservation_params:
        email: <YOUR_R2LAB_EMAIL>
        password: <YOUR_R2LAB_PASSWORD>

nodes:
  sopnode-f1:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  sopnode-f2:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  sopnode-f3:
    provisioner: pos
    roles:
      - core
      - ric

ran:
  implementation: oai
  components:
    cucp:
      type: cu_cp
      advanced_config:
        useAdditionalOptions: "--telnetsrv --telnetsrv.shrmod ci --telnetsrv.listenport 9091"  # telnet defaults to 9090, the port of the metrics-parser sidecar in the same pod

    cuup:
      type: cu_up

    du01:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.102"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 1
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f1

    du02:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.104"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 2
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f2

ric:
  implementation: oai

rus:
  jaguar:
    vendor: aw2s
    model: jaguar
    provisioner: r2lab

  panther:
    vendor: aw2s
    model: panther
    provisioner: r2lab

ues:
  qhat01:
    type: cots
    provisioner: r2lab

  qhat02:
    type: cots
    provisioner: r2lab

  qhat03:
    type: cots
    provisioner: r2lab
"""
)

> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publicly!

By default, the metamodel and deployment model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

### The AW2S configurations are referenced from: https://github.com/sopnode/oai5g-rru/blob/main/rru/gnb-config-aw2s.yaml

## **1.**   Provisioning nodes:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision_nodes()

## **2.**   Kubernetes deployment:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

## **3.**   5G core deployment:

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## **4.**   RIC deployment:

In [ ]:
from orchestrator.deployment import RICManagement

ric = RICManagement(core)

In [ ]:
await ric.deploy()

## **5.**   RAN nodes tuning:

In [ ]:
from orchestrator.deployment import MiscellaneousManagement

misc = MiscellaneousManagement(ric)

Using the `MiscellaneousManagement` management class, we will run a custom playbook to apply the necessary netplan configuration for `sopnode-f1` and `sopnode-f2` to reach the R2Lab network, as well as other CPU and NIC optimization needed for a healthy RAN deployment.

#### The tuning roles and playbook are taken directly from: https://github.com/sopnode/5g_ansible 

In [ ]:
from pathlib import Path

await misc.run_playbook(
    playbook=str(Path("tuning/ansible/playbooks/tuning.yaml").resolve()),
    roles_paths=[str(Path("tuning/ansible/roles").resolve())],
)

## **6.**   RAN deployment:

First, we are going to provision the remaining resources (RUs and UEs):

In [ ]:
prov_radio = ProvisioningManagement(misc)
await prov_radio.provision()

### Then we are going to deploy the OAI RAN control-plane and the first DU (`du01` with `jaguar`).

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(prov_radio)

In [ ]:
# Select local OAI charts that include the metrics-parser sidecar.
from importlib.resources import files

oai_charts = files("nador") / "assets" / "ran" / "oai"
for name, component in ran.deployment_model.ran.components.items():
    if (
        component.implementation == "oai"
        and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du")
        and (name in ['cucp', 'cuup'] or component.type == "du")
    ):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


### CSI logging

Both DUs (`du01` with Jaguar on `sopnode-f1`, `du02` with Panther on `sopnode-f2`) can; the CU-CP and CU-UP have no PHY, hence no SRS, and log SRS channel estimates per resource block. `enable_csi` selects a CSI logger image with the same
OAI tag for the radios of `CSI_COMPONENTS` only (every other OAI component, and all of them with an empty list, use the image without logger), forces periodic SRS and CSI-RS in the rendered configuration, and mounts `/data/csi` from the
node: each logger writes `/data/csi/csi_per_rb.csv` on its own node, where the file survives pod restarts. Run these
cells before preparing the radios.

`CSI_ENV` passes options to the logger (read when the radio starts): `CSI_GRANULARITY` (`rb` or `subcarrier`),
`CSI_SUBCARRIER_SAMPLING` (1 to 12), `CSI_ANTENNA_SELECTION` / `CSI_PORT_SELECTION` (`all` or indices) and
`CSI_FLUSH_CORE` (CPU of the flush thread, default 32). `CSI_COLLECT` sets how each experiment retrieves the CSI;
the full option tables are in `examples/monitoring/multi-gnb-2du/test.ipynb`. With a real radio the file grows by
several MB/s: experiments copy only the per-section files (`fetch_full=False`), still reading the whole file on the
radio node at the lowest priority, on its housekeeping CPUs.

In [ ]:
from nador.csi import CsiCollector, enable_csi

# CSI mode: radios that log (empty list: no CSI) and logger options (empty: defaults, RB granularity)
CSI_COMPONENTS = ['du01', 'du02']
CSI_ENV = {}  # e.g. {"CSI_GRANULARITY": "subcarrier", "CSI_SUBCARRIER_SAMPLING": "2"}
# CSI retrieval by each ExperimentRun: per-section files only (the whole file grows by several MB/s)
CSI_COLLECT = {'fetch_full': False, 'window_types': 'section', 'padding_seconds': 0, 'cpu_list': '0-4'}

enable_csi(ran, components=CSI_COMPONENTS, env=CSI_ENV)
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

# One collector per CSI component, reused by every ExperimentRun below.
CSI_COLLECTORS = [CsiCollector(ran, name, **CSI_COLLECT) for name in CSI_COMPONENTS]

#### Default CSI settings

With the CSI cells as provided (`CSI_ENV = {}`), the settings below apply. Change them in the cells named in the last column; logger settings take effect when the radios start, so set them before preparing the radios.

| Setting | Value in this notebook | Where it is set |
|---|---|---|
| Granularity (`CSI_GRANULARITY`) | `rb`: one complex value per RB | `CSI_ENV` (logger default) |
| Subcarrier sampling (`CSI_SUBCARRIER_SAMPLING`) | 1 (only used with `subcarrier`) | `CSI_ENV` (logger default) |
| Antennas / ports (`CSI_ANTENNA_SELECTION` / `CSI_PORT_SELECTION`) | all | `CSI_ENV` (logger default) |
| Writer thread CPU (`CSI_FLUSH_CORE`) | 32 (left unpinned with a warning when the CPU does not exist) | `CSI_ENV` (logger default) |
| SRS period | chosen by OAI: smallest allowed period covering TDD period x maximum number of UEs per gNB (measured in the rfsim scenarios: 160 slots, 80 ms) | OAI gNB/DU (`do_SRS: periodic`) |
| CSI file | `/data/csi/csi_per_rb.csv` on the radio node (hostPath) | `enable_csi` (`output_dir`) |
| CSI storage | data disk bind-mounted on `/data/csi` (`CSI_STORAGE` partitions) | `prepare_csi_storage` |
| Whole CSI file copied per run (`fetch_full`) | no (per-section files only) | `CSI_COLLECT` |
| Windows cut (`window_types`, `padding_seconds`) | `section`, 0 s | `CSI_COLLECT` |
| Reading CPUs on the radio node (`cpu_list`) | `0-4` (housekeeping CPUs) | `CSI_COLLECT` |
| Temporary files on the radio node (`cleanup_remote`) | removed | `CsiCollector` default |
| Clock offset correction | applied only above 0.5 s (offset and drift recorded per run) | `ExperimentRun(csi_clock_threshold_s=0.5)` |
| Live view | on `sopnode-f3`, ports `8501`, `8502`, 120 s window, 10 s refresh, live copy restarted beyond 1024 MB, visualizer v9.0 (`oai-csi-logging` `csi-v3.1`), `tail -F` on CPUs `0-4` | `CSI_LIVE`, `CSI_LIVE_PORTS` |

The R2Lab nodes boot a live image: their root filesystem is in RAM, where the CSI file would fill the memory
(5g_ansible saw the gNB pod killed after about 2 minutes). `prepare_csi_storage` bind-mounts the `csi-data`
directory of the node's data disk onto `/data/csi`, as 5g_ansible does; the partitions are those of the 5g_ansible
inventory (`storage=`). The disk is never formatted, and the mount does not survive a reboot: rerun this cell after
reprovisioning. It stops with an explicit error if a node's directory is in RAM and no partition is given.

In [ ]:
from nador.csi import prepare_csi_storage

# Data disk partition (under /dev) of each radio node whose root is in RAM; nodes on a disk need no entry
CSI_STORAGE = {'sopnode-f1': 'sda1', 'sopnode-f2': 'sda1'}
for directory, report in ((await prepare_csi_storage(ran, CSI_STORAGE)) if CSI_COMPONENTS else {}).items():
    print(f"{directory}: {report['before']}  ->  {report['after']}")

In [ ]:
await ran.prepare(components=['cucp', 'cuup'])

In [ ]:
await ran.install(components=['cucp', 'cuup'])

In [ ]:
await ran.deploy(components=['du01'])

Check logs of `du01`:

In [ ]:
print(await ran.get_logs(quiet=True))

The DU is deployed and ready for connections.

## **7.**   Connecting UEs:

## Monitoring before UE attachment

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

### Live CSI view

One Streamlit view per CSI radio runs on `sopnode-f3`, never on a radio node: it follows the radio node's CSI
file over SSH, where only `sshd` and `tail -F` run, at the lowest priority on the housekeeping CPUs `0-4`. The view
shows a sliding window (120 s by default) in local time. Start it once the radio runs (it reads the file header);
the tunnel is opened on this machine. The `du02` view starts after `du02` is deployed below.

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

# Live view on sopnode-f3, never on a radio node; housekeeping CPUs for "tail -F" on the radio node
CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024, "cpu_list": "0-4"}
CSI_LIVE_PORTS = {'du01': 8501, 'du02': 8502}
lives = globals().get("lives", {})
for name in [n for n in ['du01'] if n in CSI_COMPONENTS]:
    lives[name] = CsiLiveView(ran, live_node='sopnode-f3', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    url = await lives[name].start()
    display(Markdown(f"[CSI {name}]({url})"))

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.connect()

Check `du01` logs after UEs were connected:

In [ ]:
print(await ran.get_logs(quiet=True))

The 3 UEs are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [ ]:
ping = await prov_radio.run_ue_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], quiet=True, flatten=False
)


In [ ]:
print(ping["qhat01"])

In [ ]:
print(ping["qhat02"])

In [ ]:
print(ping["qhat03"])

## **7.**   Deploy the second DU (`du02` with `panther`):

In [ ]:
ran = RANManagement(ue)

In [ ]:
await ran.deploy(components=['du02'])

Start the live CSI view of `du02`, then check that both DU loggers write their files:

In [ ]:
if "du02" in CSI_COMPONENTS:
    lives["du02"] = CsiLiveView(ran, live_node="sopnode-f3", component="du02", port=CSI_LIVE_PORTS["du02"], **CSI_LIVE)
    display(Markdown(f"[CSI du02]({await lives['du02'].start()})"))

Check that each CSI logger runs and that its file grows on disk: the logs show the logger start, and each
`csi_per_rb.csv` must grow between the two listings (the logger flushes every 5 seconds); `df` shows the space left.

In [ ]:
from orchestrator.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "[CSI]" in line or "Init: N_RB_DL" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(
        command=["sh", "-c", "ls -l /data/csi/csi_per_rb.csv; df -h /data/csi | tail -1"],
        nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False,
    )
    for name, node in csi_nodes.items():
        print(f"{name} ({node}):\n{listing[node].strip()}")
    await asyncio.sleep(10)

#### Check `du01`'s and `du02`'s logs before the F1-handover attempt:

In [ ]:
logs = await ran.get_logs(components=["du01", "du02"], quiet=True)

In [ ]:
print(logs["du01"])

In [ ]:
print(logs["du02"])

## **8.**   Trigger F1-Handover using OAI CU-CP's telnet server command:

Get one of the UE's CU UE ID (usually they are incremental integers starting from 1)

In [ ]:
print(await ran.get_logs(components=["cucp"], quiet=True))

Let's try to handover UE with CU UE ID 1 from `du01` to `du02`.

Reference: https://github.com/OPENAIRINTERFACE/openairinterface5g/blob/develop/doc/handover-tutorial.md

In [ ]:
print(
    (
        await ran.run_command(
            command=[
                "bash",
                "-c",
                "'exec 3<>/dev/tcp/127.0.0.1/9090; "
                'printf "ci trigger_f1_ho 1\\n" >&3; '
                "IFS= read -r response <&3; "
                'printf "%s\\n" "$response"; '
                "exec 3<&-; "
                "exec 3>&-'",
            ],
            components=["cucp"],
            quiet=True,
            flatten=False,
        )
    )["cucp"]
)


The handover has been triggered.

Let's check `du02`'s logs after the handover:

In [ ]:
print(await ran.get_logs(quiet=True))

In this case, the F1-handover attempt did not work as expected. It will be debugged later. A possible cause of this failure is that both DUs were operating at the exact same frequency and might have interfered with each other.

## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline. The run also collects the CSI of each radio, cut per section under `csi/<component>/`.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.deployment import ProvisioningManagement
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ["qhat01", "qhat02", "qhat03"]
UE_INTERFACE = "wwan0"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ProvisioningManagement(preceding=ue)

iperf_setup_error = None
try:
    await run_ansible_playbook(
        playbook=[
            {
                "name": "Install iperf3 on the server host",
                "hosts": SERVER_NODE,
                "gather_facts": False,
                "become": True,
                "tasks": [
                    {
                        "ansible.builtin.apt": {
                            "name": "iperf3",
                            "state": "present",
                            "update_cache": True,
                            "cache_valid_time": 3600,
                        }
                    }
                ],
            }
        ],
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )
    await clients.run_ue_command(
        command=[
            "sh",
            "-c",
            "command -v iperf3 >/dev/null 2>&1 || (DEBIAN_FRONTEND=noninteractive apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y --no-upgrade iperf3)",
        ],
        ues=CLIENTS,
        become=True,
        quiet=True,
        flatten=False,
    )
except RuntimeError as exc:
    iperf_setup_error = str(exc)
    print(f"iperf setup failed; ping and monitoring checks will continue: {exc}")

addresses = await clients.run_ue_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], ues=CLIENTS, quiet=True, flatten=False
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [
        a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"] if a["family"] == "inet"
    ]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_ue_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        ues=[client],
        become=True,
        quiet=True,
        flatten=False,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'qhat01', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'qhat01', "direction": "downlink", "duration": DURATION},
    {"name": "ue2_uplink", "client": 'qhat02', "direction": "uplink", "duration": DURATION},
    {"name": "ue2_downlink", "client": 'qhat02', "direction": "downlink", "duration": DURATION},
    {"name": "ue3_uplink", "client": 'qhat03', "direction": "uplink", "duration": DURATION},
    {"name": "ue3_downlink", "client": 'qhat03', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    with experiment.section("ping"):
        ping = await clients.run_ue_command(
            command=["ping", "-c", "6", "11.1.1.1", "-I", UE_INTERFACE], ues=CLIENTS, quiet=True, flatten=False
        )
    experiment.save_json("ping.json", ping)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        try:
            if iperf_setup_error:
                raise RuntimeError(iperf_setup_error)
            await start_iperf_server(port)

            with experiment.section(test["name"]):
                report = await iperf_client(
                    client=test["client"],
                    direction=test["direction"],
                    duration=test["duration"],
                    port=port,
                )
        except (RuntimeError, ValueError) as exc:
            report = {"error": str(exc)}
            print(f"FAILED {test['name']}: {exc}")

        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read in the whole file, and the per-section files under
`csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("clock_offset_applied", "clock_offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before releasing the nodes: this closes the tunnels, stops the viewers and revokes their
SSH keys on the radio nodes.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

Let's disconnect the UEs for now:

In [ ]:
ue = UEManagement(ran)
await ue.disconnect(force=True)

And finally save this experiment's meta model and deployment model:

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/post-5g/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.